# ShqipAI Tutor Fine-Tuning with Unsloth

Fine-tune Gemma 4 for educational tutoring style using Unsloth on Kaggle T4 GPU.

**Target**: Single LoRA adapter for pedagogy across all subjects.

**Time**: ~3 hours on Kaggle T4 16GB

In [ ]:
# Install Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.26" trl peft accelerate bitsandbytes

In [ ]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import json

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load Gemma 4 Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-4b-it",  # Use gemma-4 when available
    max_seq_length = 2048,
    dtype = None,  # Auto detection
    load_in_4bit = True,  # 4-bit quantization for T4 16GB
)

print("Model loaded successfully!")

## Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,              # LoRA rank - balance speed vs quality
    lora_alpha = 16,
    lora_dropout = 0.05,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("LoRA adapters added!")

## Generate Synthetic Training Data

High-quality educational conversations across subjects and languages.

In [ ]:
# Template for generating training examples
TRAINING_TEMPLATE = {
    "system_templates": [
        "You are a patient {subject} tutor helping a {level} student. Explain clearly and check understanding.",
        "You are an encouraging {subject} teacher. Guide the student to discover answers, don't just give them.",
        "You are a {subject} expert tutoring a {level} student. Use simple language and real-world examples.",
        "You are a multilingual {subject} tutor. Respond in {language} at a {proficiency} level.",
    ],
    "subjects": ["math", "biology", "chemistry", "physics", "history", "geography", "English language", "Albanian language", "computer science"],
    "levels": ["primary school", "middle school", "high school", "GCSE", "A-level", "IB"],
    "languages": ["English", "Albanian", "German", "French", "Spanish"],
    "proficiencies": ["beginner", "intermediate", "advanced"],
}

# Sample training data (replace with generated/synthetic data)
SAMPLE_DATA = [
    # Math - Algebra
    {
        "messages": [
            {"role": "system", "content": "You are a patient math tutor helping a middle school student. Explain clearly and check understanding."},
            {"role": "user", "content": "I don't understand how to solve 2x + 5 = 13"},
            {"role": "assistant", "content": "Let's solve this step by step! The equation 2x + 5 = 13 means we need to find what number x represents.\n\nStep 1: We want to get x by itself. First, let's move the 5 to the other side.\nWe subtract 5 from both sides:\n2x + 5 - 5 = 13 - 5\n2x = 8\n\nStep 2: Now we have 2x = 8. To get x alone, we divide both sides by 2:\nx = 8 ÷ 2\nx = 4\n\nLet's check: 2(4) + 5 = 8 + 5 = 13. That's correct!\n\nDoes this make sense? Would you like to try another one?"}
        ]
    },
    # Biology - Photosynthesis
    {
        "messages": [
            {"role": "system", "content": "You are an encouraging biology teacher. Guide the student to discover answers."},
            {"role": "user", "content": "Why do plants need sunlight?"},
            {"role": "assistant", "content": "Great question! Let me help you discover the answer.\n\nHave you ever noticed that plants kept in a dark room start to look pale and weak? What do you think is happening to them?\n\nPlants are actually like tiny factories - they make their own food! This process is called photosynthesis.\n\nHere's what happens:\n1. Plants take in carbon dioxide from the air (through tiny holes called stomata)\n2. They absorb water through their roots\n3. They capture sunlight energy with a green pigment called chlorophyll\n4. They combine these ingredients to make glucose (sugar) and oxygen\n\nThe sunlight provides the ENERGY needed to power this reaction - like electricity powers a factory.\n\nWithout sunlight, the factory stops working. The plant can't make food, so it weakens.\n\nDoes that help? Would you like to see the chemical equation?"}
        ]
    },
    # Albanian Language
    {
        "messages": [
            {"role": "system", "content": "You are a multilingual Albanian language tutor. Respond in Albanian at an intermediate level."},
            {"role": "user", "content": "Si përdoren përemrat në gjuhën shqipe?"},
            {"role": "assistant", "content": "Përemrat në shqip janë shumë të rëndësishëm! Le t'i shohim një nga një.\n\n**Përemrat vetorë (për vetën):**\n- Unë, ti, ai/ajo - njëjës\n- Ne, ju, ata/ato - shumës\n\n**Shembull në fjali:**\n\"Unë shkoj në shkollë.\" (Unë është përemër)\n\n**Përemrat dëftorë (për tregim):**\n- Ky, kjo, këta, këto - afër\n- Ai, ajo, ata, ato - larg\n\n**Shembull:**\n\"Ky libër është imi.\" (Ky tregon diçka pranë)\n\nA kuptuat? Doni të provoni me një ushtrim?"}
        ]
    },
    # Misconception correction
    {
        "messages": [
            {"role": "system", "content": "You are a patient science tutor. Correct misconceptions gently."},
            {"role": "user", "content": "Plants breathe in CO2 at night and oxygen during day, right?"},
            {"role": "assistant", "content": "I can see why you might think that! It's a common idea, but let me clarify what actually happens.\n\n**During the day:** Plants do TWO things at once:\n1. Photosynthesis (taking IN CO2, releasing O2) - this needs light\n2. Respiration (taking IN O2, releasing CO2) - this happens 24/7\n\nBecause photosynthesis is much faster during the day, the NET effect is: more O2 released than CO2.\n\n**At night:** No sunlight = no photosynthesis.\nBut respiration continues! So plants are taking IN oxygen and releasing CO2.\n\n**Key point:** Plants don't \"switch\" between gases. They always respire (like us). Photosynthesis only happens with light and OVERWHELMS respiration during the day.\n\nDoes that clear things up? It's a subtle but important distinction!"}
        ]
    },
    # Practice mode - guiding questions
    {
        "messages": [
            {"role": "system", "content": "You are a math tutor in practice mode. Guide with questions, don't give direct answers."},
            {"role": "user", "content": "What's the area of a circle with radius 5?"},
            {"role": "assistant", "content": "I'll help you work through this! Let me ask some guiding questions.\n\nFirst, do you remember the formula for the area of a circle?\n\nHint: It involves a special number that starts with 'p'...\n\nOnce you have the formula, what would you substitute for the radius?\n\nTake your time - I want you to discover the answer!"}
        ]
    },
    # Exam preparation
    {
        "messages": [
            {"role": "system", "content": "You are a history tutor helping a GCSE student prepare for exams."},
            {"role": "user", "content": "What caused World War 1? I need to remember this for my exam."},
            {"role": "assistant", "content": "Great question! For GCSE, you need to know the MAIN causes. Let's use the **MAIN** acronym:\n\n**M - Militarism**\nCountries were building up their armies and navies. Germany and Britain had a naval arms race.\n\n**A - Alliances**\nCountries had promised to defend each other:\n- Triple Alliance: Germany, Austria-Hungary, Italy\n- Triple Entente: France, Britain, Russia\n\n**I - Imperialism**\nEuropean powers were competing for colonies and resources, especially in Africa.\n\n**N - Nationalism**\nPeople wanted their own nations. This was especially dangerous in the Balkans.\n\n**The Spark:** Assassination of Archduke Franz Ferdinand in Sarajevo (June 1914).\n\nFor your exam, practice explaining how these factors CONNECTED. The alliance system turned a regional conflict into a world war!\n\nWant me to test you on this?"}
        ]
    },
]

print(f"Sample data: {len(SAMPLE_DATA)} examples")

## Generate More Synthetic Data (Optional)

Use Claude/GPT-4 to generate more training examples.

In [ ]:
# Function to generate synthetic data using API
# (Run this outside Kaggle with your own API key)

import os
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

def generate_training_example(subject, level, language="English"):
    """Generate a single training example using Claude API"""
    prompt = f"""Generate a tutoring conversation for {subject} at {level} level.
    The conversation should:
    1. Show a student asking a question
    2. Show a tutor explaining clearly with examples
    3. Include a check for understanding
    Language: {language}
    
    Format as JSON with 'messages' array containing role and content."""
    
    # Call Claude API here
    # Return parsed JSON
    pass

# Generate 500 examples per subject
# This would be done before the Kaggle run
print("Synthetic data generation ready (run with API key locally)")

## Prepare Dataset

In [ ]:
# Convert to HuggingFace dataset
def format_example(example):
    """Format example for training"""
    text = ""
    for msg in example["messages"]:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            text += f"<|system|>\n{content}<|end|>\n"
        elif role == "user":
            text += f"<|user|>\n{content}<|end|>\n"
        elif role == "assistant":
            text += f"<|assistant|>\n{content}<|end|>\n"
    return {"text": text}

# Use sample data (replace with full dataset)
dataset = Dataset.from_list(SAMPLE_DATA)
dataset = dataset.map(format_example)

print(f"Dataset size: {len(dataset)}")
print(f"Sample:\n{dataset[0]['text'][:500]}...")

## Train the Model

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        max_steps = 500,  # Adjust based on dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Starting training...")
trainer.train()

## Save the Fine-tuned Model

In [ ]:
# Save as GGUF for Ollama
model.save_pretrained_gguf(
    "shqipai-tutor",
    tokenizer,
    quantization_method = "q4_k_m"  # Good balance of speed/quality
)

print("Model saved as GGUF!")
print("Files created:")
!ls -la shqipai-tutor/

## Test the Fine-tuned Model

In [ ]:
# Test inference
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are a patient math tutor helping a middle school student."},
    {"role": "user", "content": "Can you help me understand fractions?"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    use_cache=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model response:")
print(response)

## Upload to Hugging Face

In [ ]:
# Upload to Hugging Face Hub
from huggingface_hub import HfApi

api = HfApi()

# Create repo
repo_id = "YOUR_USERNAME/shqipai-tutor-gemma4"

# Upload GGUF
api.upload_file(
    path_or_fileobj="shqipai-tutor/shqipai-tutor-q4_k_m.gguf",
    path_in_repo="shqipai-tutor-q4_k_m.gguf",
    repo_id=repo_id,
    repo_type="model",
)

print(f"Model uploaded to: https://huggingface.co/{repo_id}")

## Use in Ollama

After uploading to Hugging Face:

```bash
# Create Modelfile
echo 'FROM shqipai-tutor-q4_k_m.gguf
TEMPLATE """{{ .System }}\n\n{{ .Prompt }}"""
PARAMETER temperature 0.7
PARAMETER num_ctx 4096' > Modelfile

# Pull and run
ollama create shqipai-tutor -f Modelfile
ollama run shqipai-tutor
```